In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate datasets evaluate

from transformers import pipeline

classifier = pipeline('sentiment-analysis', model='distilbert/distilbert-base-uncased-finetuned-sst-2-english')
print(classifier('I love you'))



Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998656511306763}]


Huggingface dataset에는 상당한 양의 데이터가 있습니다. datasets을 이용하면 손쉽게 데이터를 불러올 수 있습니다.

In [ ]:
!pip install datasets

In [ ]:
import datasets
from datasets import load_dataset

huggingface_mrpc_dataset = load_dataset('e9t/nsmc', revision="refs/convert/parquet")
print(huggingface_mrpc_dataset)

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 11.1MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

default/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.71MB            

default/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})


Train datasets의 각 컬럼에 해당하는 요소들을 몇 가지만 뜯어보며, 이것들이 제대로 짝을 지어있는지 확인

In [ ]:
train = huggingface_mrpc_dataset['train']
cols = train.column_names
cols

['id', 'document', 'label']

In [ ]:
for i in range(5):
    for col in cols:
        print(col, ":", train[col][i])
    print('\n')

id : 9976970
document : 아 더빙.. 진짜 짜증나네요 목소리
label : 0


id : 3819312
document : 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
label : 1


id : 10265843
document : 너무재밓었다그래서보는것을추천한다
label : 0


id : 9045019
document : 교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
label : 0


id : 6483659
document : 사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다
label : 1




MRPC 데이터셋을 원본 텍스트 파일에서 직접 만들어보도록 하겠습니다. 원본은 Microsoft Research 가 배포한 탭 구분 텍스트 두 개(train 4,076쌍 · test 1,725쌍)로, Huggingface 의 glue 로더가 쓰는 미러에서 받을 수 있습니다. 아래 셀로 내려받은 뒤 Huggingface dataset 형태로 가공해보겠습니다.

In [ ]:
!mkdir -p data
!wget -nc -q https://dl.fbaipublicfiles.com/senteval/senteval_data/msr_paraphrase_train.txt -P data
!wget -nc -q https://dl.fbaipublicfiles.com/senteval/senteval_data/msr_paraphrase_test.txt -P data
!wc -l data/msr_paraphrase_*.txt

   1726 data/msr_paraphrase_test.txt
   4077 data/msr_paraphrase_train.txt
   5803 total


In [ ]:
import pandas as pd
from datasets import Dataset

def parse_mrpc_file(file_path):
    """MRPC 파일을 안전하게 파싱하는 함수"""
    data = {
        'Quality': [],
        '#1 ID': [],
        '#2 ID': [],
        '#1 String': [],
        '#2 String': []
    }

    with open(file_path, 'r', encoding='utf-8') as f:
        # 헤더 스킵
        next(f)

        for line_num, line in enumerate(f, 1):
            try:
                parts = line.strip().split('\t')
                if len(parts) >= 5:
                    # 5개 컬럼으로 분할 (마지막 탭들은 모두 마지막 컬럼에 포함)
                    quality = int(parts[0])
                    id1 = int(parts[1])
                    id2 = int(parts[2])
                    string1 = parts[3]
                    string2 = '\t'.join(parts[4:])  # 나머지 모든 부분을 합침

                    data['Quality'].append(quality)
                    data['#1 ID'].append(id1)
                    data['#2 ID'].append(id2)
                    data['#1 String'].append(string1)
                    data['#2 String'].append(string2)
                else:
                    print(f"Line {line_num}: Invalid format, skipping")
            except Exception as e:
                print(f"Line {line_num}: Error {e}, skipping")

    return pd.DataFrame(data)

# 로컬 파일에서 MRPC 데이터 읽기
train_df = parse_mrpc_file('data/msr_paraphrase_train.txt')
test_df = parse_mrpc_file('data/msr_paraphrase_test.txt')

데이터 구조 확인

In [ ]:
# 데이터 구조 확인
print("Train dataset columns:", train_df.columns.tolist())
print("Train dataset shape:", train_df.shape)
print("\nFirst 5 examples:")

for i in range(5):
    row = train_df.iloc[i]
    for col in train_df.columns:
        print(f"{col}: {row[col]}")
    print('\n')

Train dataset columns: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String']
Train dataset shape: (4076, 5)

First 5 examples:
Quality: 1
#1 ID: 702876
#2 ID: 702977
#1 String: Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
#2 String: Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .


Quality: 0
#1 ID: 2108705
#2 ID: 2108831
#1 String: Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .
#2 String: Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .


Quality: 1
#1 ID: 1330381
#2 ID: 1330521
#1 String: They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .
#2 String: On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .


Quality: 0
#1 ID: 3344667
#2 ID: 3344648
#1 St

모델 학습

In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# 컬럼 이름을 Huggingface glue/mrpc 와 같게 맞춘다 (Trainer 는 'label' 컬럼을 찾고, 뒤의 transform 도 그대로 쓸 수 있다)
def to_hf_columns(df):
    return (df.rename(columns={'Quality': 'label', '#1 String': 'sentence1', '#2 String': 'sentence2'})
              [['sentence1', 'sentence2', 'label']]
              .reset_index(drop=True))

train_df_hf = to_hf_columns(train_df)
test_df_hf = to_hf_columns(test_df)

# train 데이터를 train(80%)과 validation(20%)으로 층화 분할
train_split, val_split = train_test_split(train_df_hf, test_size=0.2, random_state=42, stratify=train_df_hf['label'])

customized_mrpc_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_split, preserve_index=False),
    'validation': Dataset.from_pandas(val_split, preserve_index=False),
    'test': Dataset.from_pandas(test_df_hf, preserve_index=False),
})
print(customized_mrpc_dataset)
print(customized_mrpc_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 3260
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 816
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 1725
    })
})
{'sentence1': 'The festival kicked off yesterday one day after the Competition Commission delivered its final verdict to the Government on the proposed £ 4.1 billion merger .', 'sentence2': 'The Competition Commission delivered its verdict yesterday on the proposed merger of the two big ITV players , Carlton and Granada .', 'label': 0}


Huggingface에서 Auto Model과 이에 상응하는 tokenizer을 불러오겠습니다.

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

huggingface_tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')
huggingface_model = AutoModelForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased', num_labels = 2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer와 Model을 만들었다면 이제 토크나이징하는 방법

In [ ]:
def transform(data):
    return huggingface_tokenizer(
        data['sentence1'],
        data['sentence2'],
        truncation = True,
        padding = 'max_length',
        return_token_type_ids = False,
        )

In [ ]:
hf_dataset = huggingface_mrpc_dataset.map(transform, batched=True)

# train & validation & test split
hf_train_dataset = hf_dataset['train']
hf_val_dataset = hf_dataset['validation']
hf_test_dataset = hf_dataset['test']

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

앞에서 직접 만든 커스텀 데이터셋에도 같은 transform 을 적용

In [ ]:
custom_dataset = customized_mrpc_dataset.map(transform, batched=True)

custom_train_dataset = custom_dataset['train']
custom_val_dataset = custom_dataset['validation']
custom_test_dataset = custom_dataset['test']
print(custom_train_dataset)

Map:   0%|          | 0/3260 [00:00<?, ? examples/s]

Map:   0%|          | 0/816 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'input_ids', 'attention_mask'],
    num_rows: 3260
})


Huggingface의 Trainer을 활용해 학습을 진행

In [ ]:
import os
import numpy as np
from transformers import Trainer, TrainingArguments

output_dir = './results/mrpc'

training_arguments = TrainingArguments(
    output_dir,                          # 체크포인트가 저장될 경로
    eval_strategy="epoch",               # epoch 마다 validation 평가 (4.46 미만은 evaluation_strategy)
    learning_rate=2e-5,                  # learning rate
    per_device_train_batch_size=8,       # device 당 학습 batch size
    per_device_eval_batch_size=8,        # 평가 batch size
    num_train_epochs=3,                  # 학습 epoch 수
    weight_decay=0.01,                   # weight decay
    report_to="none",                    # wandb 등 외부 로깅 끄기 (로그인 프롬프트 방지)
)

In [ ]:
!pip install evaluate

In [ ]:
from evaluate import load
metric = load('glue', 'mrpc')

def compute_metrics(eval_pred):
    predictions,labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references = labels)

Trainer에 model, arguments, train_dataset, eval_dataset, compute_metrics를 넣고 train을 진행

In [ ]:
trainer = Trainer(
    model=huggingface_model,           # 학습시킬 model
    args=training_arguments,           # TrainingArguments을 통해 설정한 arguments
    train_dataset=hf_train_dataset,    # training dataset
    eval_dataset=hf_val_dataset,       # evaluation dataset
    compute_metrics=compute_metrics,
)
trainer.train()
print("슝~")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.377841,0.833333,0.880282
2,0.519573,0.384859,0.860294,0.902896
3,0.329309,0.516713,0.855392,0.899489


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

슝~


test 데이터셋으로 평가

In [ ]:
trainer.evaluate(hf_test_dataset)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.329309,0.580146,3,0.831304,0.876956


{'eval_loss': 0.5801458954811096,
 'eval_accuracy': 0.831304347826087,
 'eval_f1': 0.8769556025369979}

방금 학습시킨 모델은 Huggingface dataset으로 학습시켰는데요, 앞에서 우리가 만든 커스텀 데이터셋(custom_train_dataset, custom_val_dataset, custom_test_dataset)으로도 학습을 진행해보고, 결과를 비교해봅시다. 커스텀 쪽은 validation 을 train 에서 20% 떼어 만들었으므로 평가 숫자가 조금 다를 수 있습니다.

In [ ]:
# 메모리를 비워줍니다. trainer 도 모델을 참조하므로 함께 지운다.
import gc, torch
del trainer, huggingface_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Q. 커스텀 데이터셋으로 학습시켜봅시다.
huggingface_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert/distilbert-base-uncased', num_labels=2)

trainer_custom = Trainer(
    model=huggingface_model,
    args=training_arguments,
    train_dataset=custom_train_dataset,
    eval_dataset=custom_val_dataset,
    compute_metrics=compute_metrics,
)
trainer_custom.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.458738,0.806373,0.858423
2,0.502239,0.650494,0.790441,0.855696
3,0.302421,0.629243,0.810049,0.860234


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1224, training_loss=0.37406615649952607, metrics={'train_runtime': 542.649, 'train_samples_per_second': 18.023, 'train_steps_per_second': 2.256, 'total_flos': 1295531158855680.0, 'train_loss': 0.37406615649952607, 'epoch': 3.0})

In [ ]:
# Validation 데이터셋으로 평가한 뒤, test 데이터셋으로도 평가해 Huggingface dataset 결과와 비교합니다.
print(trainer_custom.evaluate())
print(trainer_custom.evaluate(custom_test_dataset))

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.302421,0.629243,3,0.810049,0.860234


{'eval_loss': 0.6292425394058228, 'eval_accuracy': 0.8100490196078431, 'eval_f1': 0.8602344454463481}


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.302421,0.587760,3,0.819130,0.865864


{'eval_loss': 0.5877595543861389, 'eval_accuracy': 0.8191304347826087, 'eval_f1': 0.8658641444539983}
